# Preprocessing


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/SpotifyAudioFeaturesApril2019.csv")

## Duplicate Management

In [3]:
df["track_id"].duplicated().sum()


np.int64(337)

In [4]:
df = (
    df.sort_values("popularity", ascending=False)
      .drop_duplicates(subset="track_id", keep="first")
)


Duplicate tracks were resolved by keeping the version with the highest popularity,
assuming it reflects the most updated engagement level.


## Outliers Management

Capping Function IQR

In [5]:
def cap_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)


In [6]:
for col in ["duration_ms", "tempo", "loudness"]:
    df[col] = cap_outliers(df[col])


Outliers were capped using the IQR method to reduce the influence of extreme values while preserving the full dataset size.

# Target Definition (Classification)

EDA showed strong skew - regression would be unstable.

## Popularity Threshold

In [7]:
df["popularity"].describe()


count    130326.000000
mean         24.130573
std          19.662458
min           0.000000
25%           7.000000
50%          21.000000
75%          38.000000
max         100.000000
Name: popularity, dtype: float64

In [8]:
threshold = df["popularity"].quantile(0.75)

df["is_popular"] = (df["popularity"] >= threshold).astype(int)


Tracks in the top 25% of popularity are labeled as popular (1),
while the rest are labeled as non-popular (0).


## Feature Selection

In [9]:
df_model = df.drop(
    columns=["track_id", "track_name", "artist_name", "popularity"]
)


Remove irrelevant columns

## Encoding of Categorical Variables

In [11]:
df_model = pd.get_dummies(
    df_model,
    columns=["key", "mode", "time_signature"],
    drop_first=True
)


Categorical variables were one-hot encoded to make them compatible with
machine learning algorithms.


## Featuring Scaling

In [12]:
X = df_model.drop("is_popular", axis=1)
y = df_model["is_popular"]


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [14]:
pd.DataFrame(X_scaled, columns=X.columns).to_csv(
    "../data/processed/X_scaled.csv", index=False
)

y.to_csv("../data/processed/y.csv", index=False)
